In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# A/Bテストフラグ
# ============================================================
TEST_A_WEIGHTED_KNN   = True    # ← 逆距離加重KNN追加
TEST_B_D2_KNN         = False
TEST_C_KNN_DISTANCE   = False
TEST_D_HIGHER_LGB_W   = False
TEST_E_DROP_RIDGE     = False
TEST_F_BAND_AREA      = False

# LGB単独（PLS/Ridgeは害と判明）
W_LGB = 1.00
W_PLS = 0.00
W_RDG = 0.00

print("=" * 60)
print("📋 A/Bテスト設定:")
print(f"  A: 逆距離加重KNN  = {TEST_A_WEIGHTED_KNN}")
print(f"  B: d2空間KNN     = {TEST_B_D2_KNN}")
print(f"  C: KNN距離特徴量  = {TEST_C_KNN_DISTANCE}")
print(f"  D: LGB重み0.80   = {TEST_D_HIGHER_LGB_W}")
print(f"  E: Ridge除外     = {TEST_E_DROP_RIDGE}")
print(f"  F: バンド面積追加  = {TEST_F_BAND_AREA}")
print(f"  重み: LGB={W_LGB}, PLS={W_PLS}, Ridge={W_RDG}")
print("=" * 60)

# ============================================================
# 1. データ読み込み
# ============================================================
print("\n📂 データ読み込み中...")
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

# ============================================================
# 2. 波数定義
# ============================================================
wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

band_water_5150 = np.where((wavenumbers >= 5000) & (wavenumbers <= 5300))[0]

print(f"📏 スペクトル次元: {len(spec_cols)}")

# ============================================================
# 3. 前処理
# ============================================================

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

# ============================================================
# 4. CVループ
# ============================================================
print(f"\n{'='*60}")
print(f"🚀 LGB単独 + 逆距離加重KNN テスト")
print(f"{'='*60}")

gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
oof_lgb = np.zeros(len(train))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    va_species = train.iloc[va_idx]['樹種'].unique()
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train:{len(tr_idx)}, valid:{len(va_idx)})")
    print(f"   検証樹種: {list(va_species)}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── SNV ──
    snv_tr = apply_snv(X_tr_raw)
    snv_va = apply_snv(X_va_raw)
    snv_te = apply_snv(X_te_raw)

    # ── SG微分 ──
    d1_tr = savgol_filter(snv_tr, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_va = savgol_filter(snv_va, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_te = savgol_filter(snv_te, window_length=15, polyorder=2, deriv=1, axis=1)

    d2_tr = savgol_filter(snv_tr, window_length=11, polyorder=2, deriv=2, axis=1)
    d2_va = savgol_filter(snv_va, window_length=11, polyorder=2, deriv=2, axis=1)
    d2_te = savgol_filter(snv_te, window_length=11, polyorder=2, deriv=2, axis=1)

    # ── 元コード特徴量 ──
    ratio_tr = (X_tr_raw[:, idx_1940] / (X_tr_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_va = (X_va_raw[:, idx_1940] / (X_va_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_te = (X_te_raw[:, idx_1940] / (X_te_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)

    std_tr = np.std(X_tr_raw, axis=1, keepdims=True)
    std_va = np.std(X_va_raw, axis=1, keepdims=True)
    std_te = np.std(X_te_raw, axis=1, keepdims=True)

    # ── PCA (10次元) ──
    pca = PCA(n_components=10, random_state=42)
    pca_tr = pca.fit_transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    # ── KNN (元コード: k=5, cosine) ──
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_tr)

    # Train (自分除外)
    dist_tr, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    knn_ymean_tr = np.mean(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)

    # Validation
    dist_va, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
    knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)

    # Test
    dist_te, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
    knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    # ── TEST_A: 逆距離加重KNN ──
    # Train (自分除外)
    w_tr = 1.0 / (dist_tr[:, 1:] + 1e-8)
    w_tr = w_tr / w_tr.sum(axis=1, keepdims=True)
    knn_weighted_tr = np.sum(w_tr * y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)

    # Validation
    w_va = 1.0 / (dist_va + 1e-8)
    w_va = w_va / w_va.sum(axis=1, keepdims=True)
    knn_weighted_va = np.sum(w_va * y_tr[ind_va], axis=1).reshape(-1, 1)

    # Test
    w_te = 1.0 / (dist_te + 1e-8)
    w_te = w_te / w_te.sum(axis=1, keepdims=True)
    knn_weighted_te = np.sum(w_te * y_tr[ind_te], axis=1).reshape(-1, 1)

    # ── LGB入力の組み立て ──
    feat_tr_lgb = np.hstack([
        snv_tr, d1_tr, pca_tr,
        knn_ymean_tr, knn_weighted_tr,  # ← 元KNN + 加重KNN
        ratio_tr, std_tr
    ])
    feat_va_lgb = np.hstack([
        snv_va, d1_va, pca_va,
        knn_ymean_va, knn_weighted_va,
        ratio_va, std_va
    ])
    feat_te_lgb = np.hstack([
        snv_te, d1_te, pca_te,
        knn_ymean_te, knn_weighted_te,
        ratio_te, std_te
    ])

    if fold == 0:
        print(f"\n  📐 LGB入力次元: {feat_tr_lgb.shape[1]}")
        print(f"     内訳: SNV({snv_tr.shape[1]}) + d1({d1_tr.shape[1]}) "
              f"+ PCA({pca_tr.shape[1]}) + KNN_mean(1) + KNN_weighted(1) "
              f"+ ratio(1) + std(1)")

    # ── LightGBM（元コードと同一パラメータ）──
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=5, num_leaves=31,
        subsample=0.8, colsample_bytree=0.3, random_state=42, verbosity=-1
    )
    lgb_model.fit(
        feat_tr_lgb, y_tr,
        eval_set=[(feat_va_lgb, y_va)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    p_va_lgb = np.expm1(lgb_model.predict(feat_va_lgb))
    p_te_lgb = np.expm1(lgb_model.predict(feat_te_lgb))

    # ── 蓄積 ──
    oof_lgb[va_idx] = p_va_lgb
    final_lgb += p_te_lgb / 5

    # ── RMSE ──
    y_va_real = np.expm1(y_va)
    rmse = np.sqrt(mean_squared_error(y_va_real, p_va_lgb))
    fold_rmses.append(rmse)
    print(f"  🌟 LGB RMSE: {rmse:.4f}")

    # ── 特徴量重要度（Fold 0のみ）──
    if fold == 0:
        imp = lgb_model.feature_importances_
        n_snv = snv_tr.shape[1]
        n_d1 = d1_tr.shape[1]
        n_pca = pca_tr.shape[1]

        cat_imp = {
            'SNV': np.sum(imp[:n_snv]),
            'd1': np.sum(imp[n_snv:n_snv+n_d1]),
            'PCA': np.sum(imp[n_snv+n_d1:n_snv+n_d1+n_pca]),
            'KNN_mean': imp[n_snv+n_d1+n_pca],
            'KNN_weighted': imp[n_snv+n_d1+n_pca+1],
            'ratio': imp[n_snv+n_d1+n_pca+2],
            'std': imp[n_snv+n_d1+n_pca+3],
        }
        total_imp = sum(cat_imp.values())
        print(f"\n  📊 特徴量重要度:")
        for name, val in sorted(cat_imp.items(), key=lambda x: -x[1]):
            pct = val / total_imp * 100
            bar = '█' * int(pct)
            print(f"     {name:15s}: {val:6.0f} ({pct:5.1f}%) {bar}")


# ============================================================
# 5. 全体評価
# ============================================================
print(f"\n{'='*60}")
print("📊 全体評価")
print(f"{'='*60}")

y_true_real = np.expm1(y_train_log)
oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_lgb))

print(f"  🌟 LGB OOF RMSE: {oof_rmse:.4f}")
print(f"  📊 Fold平均 RMSE: {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

# ── 樹種別残差 ──
print(f"\n  --- 樹種別残差 ---")
print(f"  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_lgb[mask.values]
    rmse_s = np.sqrt(np.mean((y_s - p_s)**2))
    bias_s = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {rmse_s:7.2f} {bias_s:+7.2f}")


# ============================================================
# 6. 提出ファイル
# ============================================================
final_blend = np.clip(final_lgb, 0, None)

submit[1] = final_blend
out = 'submission_lgb_weighted_knn.csv'
submit.to_csv(out, index=False, header=False)

print(f"\n✅ 提出ファイル: {out}")
print(f"📈 予測統計: min={final_blend.min():.1f}%, "
      f"median={np.median(final_blend):.1f}%, max={final_blend.max():.1f}%")

print(f"\n📌 比較対象:")
print(f"   元コードBlend:  LB = 12.647")
print(f"   LGB単独:        LB = 12.615")
print(f"   今回(+加重KNN): LB = ???  ← 提出して確認")

📋 A/Bテスト設定:
  A: 逆距離加重KNN  = True
  B: d2空間KNN     = False
  C: KNN距離特徴量  = False
  D: LGB重み0.80   = False
  E: Ridge除外     = False
  F: バンド面積追加  = False
  重み: LGB=1.0, PLS=0.0, Ridge=0.0

📂 データ読み込み中...
📏 スペクトル次元: 1555

🚀 LGB単独 + 逆距離加重KNN テスト

───────────────────────────────────────────────────────
📁 Fold 1/5  (train:940, valid:270)
   検証樹種: ['ウエンジ', 'トチ']

  📐 LGB入力次元: 3124
     内訳: SNV(1555) + d1(1555) + PCA(10) + KNN_mean(1) + KNN_weighted(1) + ratio(1) + std(1)
  🌟 LGB RMSE: 11.7561

  📊 特徴量重要度:
     d1             :   1784 ( 50.9%) ██████████████████████████████████████████████████
     SNV            :    777 ( 22.1%) ██████████████████████
     KNN_weighted   :    550 ( 15.7%) ███████████████
     KNN_mean       :    316 (  9.0%) █████████
     PCA            :     66 (  1.9%) █
     ratio          :     14 (  0.4%) 
     std            :      1 (  0.0%) 

───────────────────────────────────────────────────────
📁 Fold 2/5  (train:981, valid:229)
   検証樹種: ['チェリー', 'ヒノキ']
  🌟 LGB 